In [0]:

# ============================================================
# 0. CONFIG PATH
# ============================================================

dbutils.widgets.text("path", "")

config_path = dbutils.widgets.get("path")


# ============================================================
# 1. IMPORTS
# ============================================================

import json

import pyspark.sql.functions as F

from common_utils.logging import get_logger
from common_utils.generic_functions import safe_cast


# ============================================================
# 2. LOGGER
# ============================================================

logger = get_logger("dim_date_gold")


# ============================================================
# 3. READ CONFIG
# ============================================================

logger.info(
    "Reading Gold configuration from %s",
    config_path
)

with open(config_path, "r") as f:
    config = json.load(f)

logger.info(
    "Gold configuration loaded successfully"
)


# ============================================================
# 4. SOURCE / TARGET CONFIGURATION
# ============================================================

source_config = config["source"]
target_config = config["target"]

source_name = source_config["source_name"]

source_table = source_config["source_table"]

date_column = source_config["date_column"]

target_table = target_config["target_table"]


logger.info(
    "[%s] Source table: %s",
    source_name,
    source_table
)

logger.info(
    "[%s] Date column: %s",
    source_name,
    date_column
)

logger.info(
    "[%s] Target table: %s",
    source_name,
    target_table
)


# ============================================================
# 5. READ SOURCE TABLE
# ============================================================

logger.info(
    "[%s] Reading sales from %s",
    source_name,
    source_table
)

sales_df = spark.read.table(
    source_table
)

logger.info(
    "[%s] sales table loaded successfully",
    source_name
)


# ============================================================
# 6. TRANSFORM DATE
# ============================================================

def transform_date(df, config):

    logger.info(
        "[%s] Starting dim_date transformation",
        source_name
    )


    # --------------------------------------------------------
    # Get minimum and maximum dates
    # --------------------------------------------------------

    date_range = (
        df
        .select(
            F.min(
                F.to_date(
                    F.col(date_column)
                )
            ).alias("min_date"),

            F.max(
                F.to_date(
                    F.col(date_column)
                )
            ).alias("max_date")
        )
        .collect()[0]
    )


    min_date = date_range["min_date"]
    max_date = date_range["max_date"]


    logger.info(
        "[%s] Minimum order date: %s",
        source_name,
        min_date
    )

    logger.info(
        "[%s] Maximum order date: %s",
        source_name,
        max_date
    )


    if min_date is None or max_date is None:

        raise ValueError(
            "Unable to determine date range from sales"
        )


    # --------------------------------------------------------
    # Generate date sequence
    # --------------------------------------------------------

    date_df = (
        spark
        .createDataFrame(
            [(min_date, max_date)],
            ["min_date", "max_date"]
        )
        .select(
            F.explode(
                F.sequence(
                    F.col("min_date"),
                    F.col("max_date"),
                    F.expr("interval 1 day")
                )
            ).alias("date")
        )
    )


    # --------------------------------------------------------
    # Create date attributes
    # --------------------------------------------------------

    date_df = (
        date_df

        .withColumn(
            "date_key",
            F.date_format(
                F.col("date"),
                config["transform"]["date_key_format"]
            ).cast("int")
        )

        .withColumn(
            "day",
            F.dayofmonth(
                F.col("date")
            )
        )

        .withColumn(
            "month",
            F.month(
                F.col("date")
            )
        )

        .withColumn(
            "year",
            F.year(
                F.col("date")
            )
        )

        .withColumn(
            "quarter",
            F.quarter(
                F.col("date")
            )
        )

        .withColumn(
            "week",
            F.weekofyear(
                F.col("date")
            )
        )
    )


    # --------------------------------------------------------
    # Cast final columns
    # --------------------------------------------------------

    date_df = safe_cast(
        date_df,
        config.get("cast_columns", {})
    )


    # --------------------------------------------------------
    # Select final columns
    # --------------------------------------------------------

    date_df = date_df.select(
        *config["select_columns"]
    )


    logger.info(
        "[%s] dim_date transformation completed",
        source_name
    )

    return date_df


# ============================================================
# 7. CREATE DIM DATE
# ============================================================

logger.info(
    "[%s] Starting dim_date creation",
    source_name
)

dim_date = transform_date(
    sales_df,
    config
)


# ============================================================
# 8. VALIDATE DATA
# ============================================================

logger.info(
    "[%s] Validating dim_date",
    source_name
)

record_count = dim_date.count()

logger.info(
    "[%s] dim_date record count: %s",
    source_name,
    record_count
)

if record_count == 0:

    raise ValueError(
        "dim_date contains zero records"
    )


duplicate_count = (
    dim_date
    .groupBy("date_key")
    .count()
    .filter(F.col("count") > 1)
    .count()
)


if duplicate_count > 0:

    raise ValueError(
        "Duplicate date_key detected in dim_date"
    )


# ============================================================
# 9. DISPLAY
# ============================================================

dim_date.show()


# ============================================================
# 10. CREATE GOLD SCHEMA
# ============================================================

spark.sql("""
    CREATE SCHEMA IF NOT EXISTS
    retaildataplatform.gold
""")


# ============================================================
# 11. WRITE GOLD TABLE
# ============================================================

logger.info(
    "[%s] Writing dim_date to %s",
    source_name,
    target_table
)

(
    dim_date
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(target_table)
)


logger.info(
    "[%s] Gold table %s written successfully",
    source_name,
    target_table
)


# ============================================================
# 12. FINAL VALIDATION
# ============================================================

final_count = (
    spark.read
    .table(target_table)
    .count()
)

logger.info(
    "[%s] Final dim_date record count: %s",
    source_name,
    final_count
)

logger.info(
    "[%s] dim_date pipeline completed successfully",
    source_name
)